In [1]:
import os
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableBranch

In [2]:
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.0)

In [4]:
physics_prompt = ChatPromptTemplate.from_template("You are a smart physics professor. Question : {input}")
maths_prompt = ChatPromptTemplate.from_template("You are a brilliant mathematician. Question : {input}")
history_prompt = ChatPromptTemplate.from_template("You are an expert historian. Question : {input}")
cs_prompt = ChatPromptTemplate.from_template("You are a senior computer scientist. Question: {input}")
default_prompt = ChatPromptTemplate.from_template("You are a helpful assistant. Question: {input}")

In [13]:
classifier_prompt = ChatPromptTemplate.from_template(
    "Classify the input into exactly ONE of these words: "
    "'physics', 'maths', 'history', 'cs', or 'general'. \n\nInput: {input}"
)

classifier_chain = classifier_prompt | llm | StrOutputParser()

In [14]:
branch = RunnableBranch(
    (lambda x: "physics" in x["topic"].lower(), physics_prompt | llm | StrOutputParser()),
    (lambda x: "maths" in x["topic"].lower(), maths_prompt | llm | StrOutputParser()),
    (lambda x: "history" in x["topic"].lower(), history_prompt | llm | StrOutputParser()),
    (lambda x: "cs" in x["topic"].lower(), cs_prompt | llm | StrOutputParser()),
    default_prompt | llm | StrOutputParser()
)

In [15]:
def route_and_answer(question: str):
    topic = classifier_chain.invoke({"input": question})
    return branch.invoke({"topic": topic, "input": question})

In [12]:
print(route_and_answer("What is black body radiation?"))
print(route_and_answer("what is 2 + 2"))
print(route_and_answer("Why does every cell in our body contain DNA?"))

Black-body radiation is a fundamental concept in physics that describes the thermal radiation emitted by an object in thermal equilibrium with its environment. It's a fascinating topic that has been extensively studied and has led to significant breakthroughs in our understanding of the behavior of matter and energy.

In essence, a black body is an idealized object that absorbs all the electromagnetic radiation that falls on it, without reflecting or transmitting any of it. This means that a black body is a perfect absorber of radiation, and it has no emissivity (the ability to emit radiation) of its own.

When a black body is heated, it begins to emit radiation, which is known as black-body radiation. This radiation is a broad spectrum of electromagnetic waves, including visible light, ultraviolet (UV) radiation, infrared (IR) radiation, and even X-rays and gamma rays.

The key characteristic of black-body radiation is that it is a function of the temperature of the black body. As the